# Writing your own model, zoo-style

In this notebook we write a model from scratch: a Doyle-Fuller-Newman (DFN)
model coupled to a lumped thermal energy balance. It lives in
[`custom_model/thermal_dfn/`](custom_model/thermal_dfn), laid out exactly like a
model zoo entry, so contributing it to the zoo is a copy and a pull request.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import pybamm
import pybamm_model_zoo as zoo

## 1. The folder

```
custom_model/
├── conftest.py          # registers this folder with the zoo (workshop only)
├── test_contract.py     # runs the zoo contract on every model here
└── thermal_dfn/
    ├── model.toml
    ├── __init__.py
    ├── model.py
    ├── README.md
    ├── CITATION.bib
    ├── tests/test_thermal_dfn.py
    └── examples/run_thermal_dfn.py
```

Start with the manifest.

In [2]:
print(Path("custom_model/thermal_dfn/model.toml").read_text())

[model]
slug = "thermal_dfn"
name = "ThermalDFN"
title = "Doyle-Fuller-Newman model with a lumped thermal energy balance"
summary = "A from-scratch DFN coupled to a single cell temperature, for teaching how to write a standalone model."
# In the zoo this becomes "pybamm_model_zoo.thermal_dfn:ThermalDFN"
class = "thermal_dfn:ThermalDFN"
tier = "community"
added = "2026-10-15"
license = "BSD-3-Clause"

[[model.maintainers]]
name = "The PyBaMM Team"
github = "pybamm-team/maintainers"

[model.citation]
key = "PyBaMMConference2026"

[model.tests]
# No parameter_set: the model's own default_parameter_values supplies the new
# "Cell heat capacity [J.K-1]" parameter.
solve_time = 3600
key_variables = ["Voltage [V]", "Temperature [K]"]
skip_contract = []



## 2. The model

Open [`model.py`](custom_model/thermal_dfn/model.py) alongside this notebook.
The structure is the same as the decay model from Day 1, just with more
equations:

* **Variables** (`pybamm.Variable`) with a `domain` if they vary in space.
* **Parameters** (`pybamm.Parameter`, or the shared `self.param` object) that
  `ParameterValues` will fill in.
* **Equations**: `self.rhs` for ODEs, `self.algebraic` for algebraic
  equations, plus `self.boundary_conditions` and `self.initial_conditions`.
* **Outputs** in `self.variables`, and **events** that stop the solver.

The thermal part is three lines:

```python
heat_source = I * (ocv - voltage)
heat_loss = h * area * (T - T_amb)
self.rhs[T] = (heat_source - heat_loss) / heat_capacity
```

`"Cell heat capacity [J.K-1]"` is a new parameter, so the model supplies a
default for it in `default_parameter_values`. The zoo's contract solves your
model with its defaults, so this is required.

## 3. Register the folder with the zoo

A model inside PyBaMM's `packages/pybamm-model-zoo` is found automatically. For
a folder anywhere else, point the zoo at it as an *external* collection:

In [3]:
sys.path.insert(0, "custom_model")
zoo.refresh(external_paths=[Path("custom_model")])
zoo.list_models()

['LinearisedSPM', 'MultiLayer3DThermalSPM', 'ThermalDFN']

## 4. Run it

In [4]:
ThermalDFN = zoo.load("ThermalDFN")
model = ThermalDFN()
experiment = pybamm.Experiment(["Discharge at 1C until 2.5V", "Rest for 30 minutes"])
sim = pybamm.Simulation(model, experiment=experiment)
sol = sim.solve()
sol.plot(
    [
        "Voltage [V]",
        "Temperature [C]",
        "Heat source [W]",
        "Electrolyte concentration [mol.m-3]",
    ]
)

interactive(children=(FloatSlider(value=0.0, description='t', max=1.4902719458214562, step=0.01490271945821456…

## 5. Validate it

A model that runs is not a model that is right. Two checks:

1. With a huge heat capacity the temperature cannot change, so the model must
   reduce to PyBaMM's own DFN.
2. With no cooling, all the heat generated must end up warming the cell.

In [5]:
parameter_values = model.default_parameter_values
parameter_values["Cell heat capacity [J.K-1]"] = 1e12
isothermal = pybamm.Simulation(ThermalDFN(), parameter_values=parameter_values).solve(
    [0, 3000]
)
reference = pybamm.Simulation(
    pybamm.lithium_ion.BasicDFN(), parameter_values=pybamm.ParameterValues("Chen2020")
).solve([0, 3000])
t = np.linspace(0, 3000, 50)
error = np.abs(isothermal["Voltage [V]"](t) - reference["Voltage [V]"](t)).max()
print(f"max voltage difference: {error * 1000:.3f} mV")

max voltage difference: 0.010 mV


In [6]:
parameter_values = model.default_parameter_values
parameter_values["Total heat transfer coefficient [W.m-2.K-1]"] = 0
adiabatic = pybamm.Simulation(ThermalDFN(), parameter_values=parameter_values).solve(
    [0, 3000]
)
t = np.linspace(0, 3000, 3001)
stored = parameter_values["Cell heat capacity [J.K-1]"] * (
    adiabatic["Temperature [K]"](3000) - adiabatic["Temperature [K]"](0)
)
generated = np.trapezoid(adiabatic["Heat source [W]"](t), t)
print(f"heat stored {stored:.1f} J, heat generated {generated:.1f} J")

heat stored 2987.5 J, heat generated 2987.4 J


These checks are exactly what [`tests/test_thermal_dfn.py`](custom_model/thermal_dfn/tests/test_thermal_dfn.py)
pins. How does our model compare with PyBaMM's built-in lumped thermal DFN,
which also includes reversible heat?

In [7]:
core = pybamm.Simulation(
    pybamm.lithium_ion.DFN({"thermal": "lumped"}),
    parameter_values=pybamm.ParameterValues("Chen2020"),
    experiment=experiment,
).solve()
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for solution, temperature, label in [
    (sol, "Temperature [C]", "ThermalDFN"),
    (core, "Volume-averaged cell temperature [C]", "DFN (lumped thermal)"),
]:
    t = solution.t
    ax[0].plot(t / 60, solution["Voltage [V]"](t), label=label)
    ax[1].plot(t / 60, solution[temperature](t), label=label)
ax[0].set_ylabel("Voltage [V]")
ax[1].set_ylabel("Temperature [C]")
for a in ax:
    a.set_xlabel("Time [min]")
ax[1].legend()

## 6. Run the zoo contract and your tests

The contract checks every model in the folder for free; your tests check the
physics. From a terminal in `training/day_2/custom_model` run `pytest`, or:

In [8]:
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"],
    cwd="custom_model",
    capture_output=True,
    text=True,
)
print(result.stdout[-1500:])

...........                                                              [100%]
11 passed in 2.90s



## 7. Contribute it to the zoo

When your model is ready, read the [model zoo policy](https://github.com/pybamm-team/PyBaMM/blob/main/packages/pybamm-model-zoo/POLICY.md), then in a fork of [PyBaMM](https://github.com/pybamm-team/PyBaMM):

```bash
nox -s zoo-new -- --slug thermal_dfn --name ThermalDFN --author "Your Name" --github yourhandle
```

This renders the template into `packages/pybamm-model-zoo/src/pybamm_model_zoo/thermal_dfn/`
and adds you to `CODEOWNERS`. Then:

1. Copy in `model.py`, `README.md`, `CITATION.bib`, `tests/` and `examples/`,
   and carry over the fields of your `model.toml`.
2. Change the import path: `thermal_dfn` becomes `pybamm_model_zoo.thermal_dfn`
   (in `class` in the manifest and in `__init__.py`).
3. `nox -s zoo-docs` to generate the docs page, then
   `nox -s zoo -- --zoo-model=thermal_dfn` to run everything.
4. Add a line to the zoo's `CHANGELOG.md` and open a pull request.

## Exercises

1. Add **reversible heat**, $I T \frac{\partial U}{\partial T}$, to the heat
   source. Use `param.n.prim.dUdT(sto)` and `param.p.prim.dUdT(sto)`. Does the
   comparison with the core lumped model improve?
2. Make the cooling a **new parameter**, e.g. a fan that switches on above
   40 °C (hint: `pybamm.sigmoid`), give it a default, and re-run the contract.
3. Write a physics test for your change, then run `pytest` in `custom_model`.
4. Start your own model: copy `thermal_dfn/` to a new folder, change the slug,
   name, and class, and make the contract pass.